# Theta Optimizer

This notebook demonstrates VGG11 trained on CIFAR-10 using the **Theta** optimizer — a minimal wrapper that injects truncated [Lomax (Type-II Pareto)](https://en.wikipedia.org/wiki/Lomax_distribution) heavy-tailed noise into any PyTorch optimizer.

### Background

Injection of multiplicative, anisotropic heavy-tailed stochastic gradient noise (SGN) with subsequent truncation helps SGD escape from sharp minima, improving generalization ([Wang, Oh & Rhee, 2021](https://arxiv.org/abs/2102.04297); [Wahg & Rhee, 2025](https://arxiv.org/abs/2510.20905)).

The core heavy-tailed gradient update is:
$$
g_{\text{heavy}} = g_{SB} + Z \cdot (g_{SB} - g_{LB})
$$

where $Z \sim \text{Lomax}(\alpha, \lambda)$ is a heavy-tailed noise sample, $g_{SB}$ is a small-batch gradient, and $g_{LB}$ is a large-batch gradient (approximating the full gradient).

Note that this heavy-tail noise-injected stochastic gradient is truncated (i.e., gradient norm clipping) before being used for an update.

### Variant
With **tail-gating**, noise is injected only when the sampled CDF value exceeds a threshold:
$$
Z \cdot \mathbf{I}\{\operatorname{CDF}(Z) \ge \mathtt{tail\_prob}\}
$$

In [ ]:
import time
from itertools import cycle

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

from tqdm.auto import tqdm

from theta import Theta, NoiseStep

In [ ]:
torch.manual_seed(5959)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Uncomment the lines below for a quick demo run (smaller subset).
# train_dataset = Subset(train_dataset, range(512))
# test_dataset = Subset(test_dataset, range(512))

train_loader_sb = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
train_loader_lb = DataLoader(train_dataset, batch_size=256, shuffle=True, drop_last=True)
train_loader_sb_a = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)

train_eval_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)  # for sharpness
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"train={len(train_dataset)}, test={len(test_dataset)}")

In [ ]:
def build_vgg11(num_classes=10):
    model = models.vgg11(weights=None)
    model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    model.classifier = nn.Linear(512, num_classes)
    return model

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
def batch_to_device(batch, device):
    x, y = batch
    return x.to(device), y.to(device)


def loss_on_batch(model, batch, criterion, device):
    x, y = batch_to_device(batch, device)
    logits = model(x)
    loss = criterion(logits, y)
    acc = logits.argmax(dim=1).eq(y).float().mean().item() * 100.0
    return loss, acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        x, y = batch_to_device(batch, device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)
        total_correct += logits.argmax(dim=1).eq(y).sum().item()
        total += y.size(0)
    model.train()
    return total_loss / total, 100.0 * total_correct / total


def expected_sharpness(model, loader, criterion, device, delta=0.01, num_samples=5):
    model.eval()
    base_loss, _ = evaluate(model, loader, criterion, device)
    params = [p for p in model.parameters() if p.requires_grad]
    total_diff = 0.0

    for _ in range(num_samples):
        noises = []
        with torch.no_grad():
            for p in params:
                noise = torch.randn_like(p).mul_(delta)
                p.add_(noise)
                noises.append(noise)

        perturbed_loss, _ = evaluate(model, loader, criterion, device)
        total_diff += abs(perturbed_loss - base_loss)

        with torch.no_grad():
            for p, noise in zip(params, noises):
                p.sub_(noise)

    model.train()
    return total_diff / num_samples

In [ ]:
MAX_STEPS = 30_000
NOISE_STOP_STEP = 25_000
LEARNING_RATE = 0.01
CLIP_THRESHOLD = 0.25
TAIL_PROBABILITY = 0.9

## 1) Vanilla SGD Baseline

Standard SGD with small batch (batch size 32), no noise injection or gradient clipping.

In [ ]:
sb_model = build_vgg11().to(device)
sb_optimizer = optim.SGD(sb_model.parameters(), lr=LEARNING_RATE)

data_iter = cycle(train_loader_sb)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="SB")
for step in pbar:
    loss, acc = loss_on_batch(sb_model, next(data_iter), criterion, device)
    sb_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    sb_optimizer.step()
    pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc:.1f}")

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(sb_model, train_eval_loader, criterion, device)
test_loss, test_acc = evaluate(sb_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

## 2) Theta — Always Inject Noise

$$
g_{\text{heavy}} = g_{SB} + Z \cdot (g_{SB} - g_{LB}) = g_{SB} + Z \cdot g_{SB} - Z \cdot g_{LB}
$$

With `tail_prob=0.0`, every step injects Lomax noise (Method 2 variant: `g_SB = g_SB_a`, i.e., the current batch is reused).

**Theta training-loop protocol:**
```python
noise = optimizer.sample_noise()     # 1. sample Z from Lomax
optimizer.zero_grad()                # 2. clear gradients
if noise.active:                     # 3. conditionally inject noise
    ((-z) * loss_minus).backward()   #    accumulate -Z * g_LB
    ((z) * loss).backward()          #    accumulate +Z * g_SB  (retain_graph!)
loss.backward()                      # 4. accumulate g_SB (the "anchor")
torch.nn.utils.clip_grad_norm_(      # 5. truncate gradient norm
    model.parameters(), 
    max_norm=...
)
optimizer.step()                     # 6. step
```

In [ ]:
theta_model = build_vgg11().to(device)
theta_optimizer = Theta(
    theta_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=0.0,
)

sb_iter = cycle(train_loader_sb)    # g_SB (reused as current and plus)
lb_iter = cycle(train_loader_lb)    # g_LB (minus direction)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="Theta (p=0.0)")
for step in pbar:
    use_noise = step < NOISE_STOP_STEP
    noise = theta_optimizer.sample_noise(use_noise=use_noise)
    z = noise.value

    # forward pass on current (small) batch
    loss, acc = loss_on_batch(theta_model, next(sb_iter), criterion, device)

    theta_optimizer.zero_grad()

    if noise.active:
        # minus direction: -Z * g_LB
        loss_minus, _ = loss_on_batch(theta_model, next(lb_iter), criterion, device)
        ((-z) * loss_minus).backward()

        # plus direction: +Z * g_SB (reuse current batch loss)
        (z * loss).backward(retain_graph=True)

    # anchor gradient: g_SB
    loss.backward()
    theta_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(theta_model, train_eval_loader, criterion, device)
test_loss, test_acc = evaluate(theta_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

## 3) Theta — Tail-Gated Noise

$$
g_{\text{heavy}} = g_{SB} + Z \cdot \mathbf{I}\{\operatorname{CDF}(Z) \ge 0.9\} \cdot (g_{SB} - g_{LB})
$$

With `tail_prob=0.9`, noise is injected only when the sampled $Z$ falls in the top 10% of the Lomax distribution. Most steps reduce to vanilla clipped SGD, saving the cost of extra forward/backward passes while retaining the escape-from-sharp-minima benefit.

In [ ]:
theta_tg_model = build_vgg11().to(device)
theta_tg_optimizer = Theta(
    theta_tg_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=TAIL_PROBABILITY,
)

sb_iter = cycle(train_loader_sb)    # g_SB (current and plus)
lb_iter = cycle(train_loader_lb)    # g_LB (minus direction)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc=f"Theta (p={TAIL_PROBABILITY})")
for step in pbar:
    use_noise = step < NOISE_STOP_STEP
    noise = theta_tg_optimizer.sample_noise(use_noise=use_noise)
    z = noise.value

    # forward pass on current (small) batch
    loss, acc = loss_on_batch(theta_tg_model, next(sb_iter), criterion, device)

    theta_tg_optimizer.zero_grad()

    if noise.active:
        # minus direction: -Z * g_LB
        loss_minus, _ = loss_on_batch(theta_tg_model, next(lb_iter), criterion, device)
        ((-z) * loss_minus).backward()

        # plus direction: +Z * g_SB (reuse current batch loss)
        (z * loss).backward(retain_graph=True)

    # anchor gradient: g_SB
    loss.backward()
    theta_tg_optimizer.step()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{z:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(theta_tg_model, train_eval_loader, criterion, device)
test_loss, test_acc = evaluate(theta_tg_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

## Summary

| Experiment | `tail_prob` | Noise | Clip | Expected behavior |
|---|---|---|---|---|
| 1) Vanilla SGD | — | ✗ | ✗ | Baseline |
| 2) Theta (always) | 0.0 | ✓ | ✓ | Lower sharpness, better/similar accuracy |
| 3) Theta (tail-gated) | 0.9 | ✓ (sparse) | ✓ | Similar quality, much faster |